# Context-Aware Image Quality Assessment (IQA) — Phase 4 Research Demonstration
**Interactive Walkthrough for Academic Supervisors and Technical Reviewers**

This notebook demonstrates the Phase 4 multimodal pipeline without needing a graphical web interface:
1. **Semantic Localization**: Deep COCO person and RetinaFace detection.
2. **Local Feature Extraction**: High-resolution face/subject sharpness & motion blur anisotropy.
3. **Global Context Extraction**: Whole-frame composition and Subject-to-Background Depth-of-Field (bokeh) ratio.
4. **Multimodal Fusion & Penalties**: Decision boundary calibration with defect penalty modeling.
5. **Uncertainty & Confidence**: Calibrated error margin quantification.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Add workspace root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import Phase4Pipeline, PipelineConfig
from src.visualization.annotator import ImageAnnotator
from src.ui.data_manager import DataManager

pipeline = Phase4Pipeline()
config = PipelineConfig(threshold=55.0, w_local=0.65, w_global=0.35)
print("Phase 4 Pipeline initialized successfully!")

## 1. Case Study: Sharp Portrait with Shallow Depth-of-Field (`bday_01`)
- **Traditional IQA Limitation**: Baseline BRISQUE penalizes soft bokeh background as general frame blur.
- **Context-Aware Behavior**: Recognizes intentional subject isolation, calculates $DOF > 2.0\times$, and awards high quality.

In [ ]:
img_path = PROJECT_ROOT / "data/raw/event_01_birthday/burst_01/DSC_5108.jpg"
img = Image.open(img_path).convert("RGB")
result = pipeline.run(img, config)

annotated = ImageAnnotator.draw_annotations(img, result.localization)

plt.figure(figsize=(10, 7))
plt.imshow(annotated)
plt.title(f"bday_01 — Score: {result.unified_score:.1f} / 100 ({result.classification}) | Conf: {result.overall_confidence_pct:.1f}%")
plt.axis("off")
plt.show()

print(f"Local Technical Score:  {result.s_local:.2f} (Face Sharpness: {result.local_features.get('face_sharpness', 0):.1f})")
print(f"Global Context Score:   {result.s_global:.2f} (DOF Separation:  {result.global_features.get('dof_separation_ratio', 1.0):.2f}x)")
print(f"Final Unified Score:    {result.unified_score:.2f} [{result.classification}]")

## 2. Case Study: Severe Framing Anomaly — Head Cropped Out (`bday_44`)
- **Traditional IQA Limitation**: Misses the fact that the subject's head is completely severed at the image border.
- **Context-Aware Behavior**: Semantic localizer flags `HEAD_CROPPED`, capping the score at $\le 25.0$ (`Review`).

In [ ]:
defect_path = PROJECT_ROOT / "data/raw/event_01_birthday/burst_02/DSC_5216.jpg"
defect_img = Image.open(defect_path).convert("RGB")
defect_res = pipeline.run(defect_img, config)

annotated_defect = ImageAnnotator.draw_annotations(defect_img, defect_res.localization)

plt.figure(figsize=(10, 7))
plt.imshow(annotated_defect)
plt.title(f"bday_44 — Score: {defect_res.unified_score:.1f} ({defect_res.classification}) | Framing Penalty: -{defect_res.framing_penalty_pts:.1f} pts")
plt.axis("off")
plt.show()

print(f"Framing Anomaly:        {defect_res.localization.get('framing_anomaly')}")
print(f"Framing Penalty:        -{defect_res.framing_penalty_pts:.2f} pts (Score capped <= 25.0)")
print(f"Final Unified Score:    {defect_res.unified_score:.2f} [{defect_res.classification}]")

## 3. Case Study: Directional Motion Blur (`bday_17`)
- **Context-Aware Behavior**: Gradient anisotropy exceeds $0.45$, applying penalty deduction and correctly culling the burst photo.

In [ ]:
blur_path = PROJECT_ROOT / "data/raw/event_01_birthday/burst_02/DSC_5092.jpg"
blur_img = Image.open(blur_path).convert("RGB")
blur_res = pipeline.run(blur_img, config)

print(f"Motion Blur Metric:     {blur_res.local_features.get('motion_blur_metric'):.3f} (Threshold: 0.450)")
print(f"Motion Blur Penalty:    -{blur_res.motion_penalty_pts:.2f} pts")
print(f"Final Unified Score:    {blur_res.unified_score:.2f} [{blur_res.classification}]")